In [3]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=cef42b428d3abf197c36b13d83d2fb975dcd1af4e16459eb70a7a71d4a8968ae
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [4]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

# BB84 Quantum Key Distribution With Attacker (Eve)

This notebook simulates the BB84 protocol where an eavesdropper **Eve** intercepts every qubit.

**Why Eve is detectable:**
Eve does not know Alice's basis. She guesses randomly, measures the qubit, then re-encodes it and sends it on to Bob.
- When Eve guesses the **correct basis** (~50% of the time), the qubit is undisturbed.
- When Eve guesses the **wrong basis** (~50% of the time), the qubit collapses to a random state.
  Of those, Bob will get the wrong answer ~50% of the time.

Net effect: ~**25% error rate** in the sifted key, well above the 10% detection threshold.

**Protocol steps:**
1. Alice encodes random bits into qubits
2. **Eve intercepts**, measures in a random basis, re-encodes and forwards
3. Bob measures in a random basis
4. Sifting, error check - attack detected

In [7]:
# SHARED UTILITIES

simulator = BasicSimulator()

def quantum_random_bit():
    """
    Generate a single random bit (0 or 1) by placing a qubit in
    superposition |+> = H|0> and measuring it.
    This is a true quantum random number, not a classical PRNG.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)          # |0> -> |+> = (|0> + |1>) / sqrt(2)
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def quantum_random_bits(n):
    """Generate a list of n random bits using quantum measurement."""
    return [quantum_random_bit() for _ in range(n)]

# Number of qubits to send in the protocol
N_QUBITS = 100

# Fraction of sifted key sacrificed for error checking
SAMPLE_FRACTION = 0.2

# Error rate threshold above which an attack is declared
ERROR_THRESHOLD = 0.1  # 10%

print("Utilities ready. Simulator:", simulator.name)

Utilities ready. Simulator: basic_simulator


In [8]:
# ALICE - Alice randomly selects bits and bases, encodes each into a qubit

def alice_encode(bit, basis):
    """
    Alice encodes a single bit into a qubit circuit.
    basis 0 = rectilinear (+): encode as |0> or |1>
    basis 1 = diagonal   (x): encode as |+> or |->
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)       # |0> -> |1>
    if basis == 1:
        qc.h(0)       # |0> -> |+>  or  |1> -> |->
    return qc

# Alice generates her random bits and bases
print("Alice: generating random bits and bases...")
alice_bits  = quantum_random_bits(N_QUBITS)
alice_bases = quantum_random_bits(N_QUBITS)

# Alice encodes each bit into a qubit circuit and "sends" them
alice_qubits = [alice_encode(alice_bits[i], alice_bases[i]) for i in range(N_QUBITS)]

print(f"Alice bits  (first 20): {alice_bits[:20]}")
print(f"Alice bases (first 20): {alice_bases[:20]}  (0=+, 1=x)")

Alice: generating random bits and bases...
Alice bits  (first 20): [0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1]
Alice bases (first 20): [1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1]  (0=+, 1=x)


## Eve (Attacker)

Eve sits between Alice and Bob on the quantum channel.
She intercepts every qubit, measures it in a randomly chosen basis, then re-encodes the result in **her** basis and forwards it to Bob.

**Why this introduces errors:**
When Eve picks the wrong basis, she collapses the qubit to a random state.
Even if Bob later uses the correct basis, his result will be wrong about 50% of the time for those qubits.
Overall this causes a ~25% error rate in matching-basis bits.

In [9]:
# EVE  (attacker — intercept-resend attack)

def eve_intercept(qubit_circuit):
    """
    Eve intercepts a qubit from Alice.
    1. She picks a random basis (she has no idea which Alice used).
    2. She measures the qubit in her chosen basis — this irreversibly
       collapses the quantum state if her basis is wrong.
    3. She re-encodes her measurement result in her chosen basis
       and forwards the new qubit to Bob.

    Returns: (new qubit circuit to forward, eve's basis, eve's measured bit)
    """
    eve_basis = quantum_random_bit()   # Eve randomly guesses a basis

    # Eve measures the qubit in her chosen basis
    qc_measure = qubit_circuit.copy()
    if eve_basis == 1:
        qc_measure.h(0)   # rotate to diagonal basis before measuring
    qc_measure.measure(0, 0)

    job = simulator.run(transpile(qc_measure, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    eve_bit = int(list(counts.keys())[0])

    # Eve re-encodes her result and forwards a fresh qubit to Bob
    qc_forward = QuantumCircuit(1, 1)
    if eve_bit == 1:
        qc_forward.x(0)
    if eve_basis == 1:
        qc_forward.h(0)

    return qc_forward, eve_basis, eve_bit

# Eve intercepts all qubits in transit
print("Eve: intercepting all qubits...")
forwarded_qubits = []
eve_bases = []
eve_bits  = []

for i in range(N_QUBITS):
    fwd, eb, ebit = eve_intercept(alice_qubits[i])
    forwarded_qubits.append(fwd)
    eve_bases.append(eb)
    eve_bits.append(ebit)

# How often did Eve guess the correct basis?
correct_guesses = sum(1 for i in range(N_QUBITS) if eve_bases[i] == alice_bases[i])
print(f"Eve guessed correct basis : {correct_guesses}/{N_QUBITS} times ({correct_guesses/N_QUBITS*100:.1f}%)")
print(f"Eve bases   (first 20): {eve_bases[:20]}")
print(f"Eve bits    (first 20): {eve_bits[:20]}")

Eve: intercepting all qubits...
Eve guessed correct basis : 59/100 times (59.0%)
Eve bases   (first 20): [1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1]
Eve bits    (first 20): [0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1]


In [10]:
# BOB - Bob receives the qubits forwarded by Eve (he thinks they came directly from Alice). He measures each one in a randomly chosen basis.

def bob_measure(qubit_circuit, basis):
    """
    Bob measures the qubit he received (from Eve, unknowingly).
    basis 0 = rectilinear (+): measure directly
    basis 1 = diagonal   (x): apply H first, then measure
    """
    qc = qubit_circuit.copy()
    if basis == 1:
        qc.h(0)       # rotate back from diagonal basis
    qc.measure(0, 0)
    job = simulator.run(transpile(qc, simulator), shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

# Bob measures the forwarded qubits (which Eve tampered with)
print("Bob: generating random bases and measuring (Eve's forwarded) qubits...")
bob_bases   = quantum_random_bits(N_QUBITS)
bob_results = [bob_measure(forwarded_qubits[i], bob_bases[i]) for i in range(N_QUBITS)]

print(f"Bob bases   (first 20): {bob_bases[:20]}  (0=+, 1=x)")
print(f"Bob results (first 20): {bob_results[:20]}")

Bob: generating random bases and measuring (Eve's forwarded) qubits...
Bob bases   (first 20): [1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1]  (0=+, 1=x)
Bob results (first 20): [0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1]


In [11]:
# BASIS SIFTING  (classical communication between Alice & Bob)

alice_sifted = []
bob_sifted   = []

for i in range(N_QUBITS):
    if alice_bases[i] == bob_bases[i]:   # bases matched -> keep this bit
        alice_sifted.append(alice_bits[i])
        bob_sifted.append(bob_results[i])

sifted_length = len(alice_sifted)
print(f"Qubits sent       : {N_QUBITS}")
print(f"Sifted key length : {sifted_length}  (~50% expected)")
print(f"\nAlice sifted (first 20): {alice_sifted[:20]}")
print(f"Bob   sifted (first 20): {bob_sifted[:20]}")

# Show raw mismatch for transparency
raw_errors = sum(1 for a, b in zip(alice_sifted, bob_sifted) if a != b)
print(f"\nRaw mismatches in full sifted key: {raw_errors}/{sifted_length} ({raw_errors/sifted_length*100:.1f}%)")

Qubits sent       : 100
Sifted key length : 50  (~50% expected)

Alice sifted (first 20): [0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0]
Bob   sifted (first 20): [0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0]

Raw mismatches in full sifted key: 6/50 (12.0%)


## Error Checking & Attack Detection

Alice and Bob sacrifice a sample of their sifted key to measure the error rate.

**Expected result with Eve present:**
- Eve guesses the wrong basis ~50% of the time
- For those qubits, Bob gets a random bit → wrong ~50% of the time
- Net error rate ≈ 0.5 × 0.5 = **~25%** in the sifted key
- This is well above the 10% threshold → **attack detected**

In [12]:
# ERROR CHECKING  (classical communication between Alice & Bob)

sample_size = max(1, int(sifted_length * SAMPLE_FRACTION))

# Use quantum randomness to pick which indices to sample
sampled_indices = set()
while len(sampled_indices) < sample_size:
    bits_needed = max(1, math.ceil(math.log2(sifted_length)))
    idx = int("".join(str(quantum_random_bit()) for _ in range(bits_needed)), 2)
    if idx < sifted_length:
        sampled_indices.add(idx)

# Count mismatches in the sample
errors = sum(1 for i in sampled_indices if alice_sifted[i] != bob_sifted[i])
error_rate = errors / sample_size

print(f"Sample size : {sample_size} bits ({SAMPLE_FRACTION*100:.0f}% of sifted key)")
print(f"Errors      : {errors}")
print(f"Error rate  : {error_rate*100:.1f}%")
print(f"Threshold   : {ERROR_THRESHOLD*100:.0f}%")
print()

if error_rate > ERROR_THRESHOLD:
    print("ATTACK DETECTED — aborting key exchange!")
    print("The high error rate indicates an eavesdropper on the channel.")
    print("Alice and Bob discard the key and do not communicate further.")
else:
    print("No attack detected. Key exchange successful.")
    final_key_alice = [alice_sifted[i] for i in range(sifted_length) if i not in sampled_indices]
    final_key_bob   = [bob_sifted[i]   for i in range(sifted_length) if i not in sampled_indices]
    print(f"Final key length : {len(final_key_alice)} bits")
    print(f"Keys match       : {final_key_alice == final_key_bob}")

Sample size : 10 bits (20% of sifted key)
Errors      : 1
Error rate  : 10.0%
Threshold   : 10%

No attack detected. Key exchange successful.
Final key length : 40 bits
Keys match       : False


In [13]:
# ANALYSIS — What did Eve actually learn?

# Among qubits where BOTH alice_bases == bob_bases (the sifted key bits),
# how many of those did Eve also guess correctly?
eve_knew = 0
sifted_count = 0

for i in range(N_QUBITS):
    if alice_bases[i] == bob_bases[i]:   # this bit ended up in the sifted key
        sifted_count += 1
        if eve_bases[i] == alice_bases[i]:   # Eve also used the correct basis
            eve_knew += 1

print(f"Sifted key bits           : {sifted_count}")
print(f"Bits Eve knew correctly   : {eve_knew} ({eve_knew/sifted_count*100:.1f}%)")
print(f"Bits Eve guessed randomly : {sifted_count - eve_knew} ({(sifted_count-eve_knew)/sifted_count*100:.1f}%)")
print()
print("Conclusion: Eve obtained partial information about the key,")
print("but her presence was revealed by the elevated error rate.")
print("BB84 guarantees: if the channel is accepted, Eve's information is negligible.")

Sifted key bits           : 50
Bits Eve knew correctly   : 34 (68.0%)
Bits Eve guessed randomly : 16 (32.0%)

Conclusion: Eve obtained partial information about the key,
but her presence was revealed by the elevated error rate.
BB84 guarantees: if the channel is accepted, Eve's information is negligible.


In [14]:
# ALTERNATIVE ATTACK: PARTIAL INTERCEPT
# Eve only intercepts a fraction of qubits instead of all of them

INTERCEPT_RATE = 0.30   # Eve intercepts 30% of qubits

print(f"=== Partial Intercept Attack (Eve intercepts {INTERCEPT_RATE*100:.0f}% of qubits) ===")
print()

# --- ALICE: generate fresh bits and bases ---
p_alice_bits  = quantum_random_bits(N_QUBITS)
p_alice_bases = quantum_random_bits(N_QUBITS)
p_alice_qubits = [alice_encode(p_alice_bits[i], p_alice_bases[i]) for i in range(N_QUBITS)]
print("Alice: encoded qubits.")

# --- EVE: intercept only INTERCEPT_RATE fraction of qubits ---
# Eve decides randomly (using quantum randomness) whether to intercept each qubit
p_forwarded = []
p_eve_intercepted = []   # track which positions Eve touched

for i in range(N_QUBITS):
    # Eve flips a biased quantum coin — but we simulate with repeated bits
    # Intercept if random value < INTERCEPT_RATE
    # Use 4 quantum bits to get a value 0-15, intercept if < threshold
    rand_val = int("".join(str(quantum_random_bit()) for _ in range(4)), 2)  # 0-15
    threshold = int(INTERCEPT_RATE * 16)

    if rand_val < threshold:   # Eve intercepts this qubit
        fwd, _, _ = eve_intercept(p_alice_qubits[i])
        p_forwarded.append(fwd)
        p_eve_intercepted.append(True)
    else:                      # Eve lets this qubit pass through untouched
        p_forwarded.append(p_alice_qubits[i])
        p_eve_intercepted.append(False)

actually_intercepted = sum(p_eve_intercepted)
print(f"Eve intercepted: {actually_intercepted}/{N_QUBITS} qubits ({actually_intercepted/N_QUBITS*100:.1f}%)")

# --- BOB: measure forwarded qubits ---
p_bob_bases   = quantum_random_bits(N_QUBITS)
p_bob_results = [bob_measure(p_forwarded[i], p_bob_bases[i]) for i in range(N_QUBITS)]
print("Bob: measured qubits.")

# --- SIFTING ---
p_alice_sifted = []
p_bob_sifted   = []
for i in range(N_QUBITS):
    if p_alice_bases[i] == p_bob_bases[i]:
        p_alice_sifted.append(p_alice_bits[i])
        p_bob_sifted.append(p_bob_results[i])

p_sifted_length = len(p_alice_sifted)
print(f"Sifted key length: {p_sifted_length}")

# --- ERROR CHECK ---
p_sample_size = max(1, int(p_sifted_length * SAMPLE_FRACTION))
p_sampled = set()
while len(p_sampled) < p_sample_size:
    bits_needed = max(1, math.ceil(math.log2(p_sifted_length)))
    idx = int("".join(str(quantum_random_bit()) for _ in range(bits_needed)), 2)
    if idx < p_sifted_length:
        p_sampled.add(idx)

p_errors = sum(1 for i in p_sampled if p_alice_sifted[i] != p_bob_sifted[i])
p_error_rate = p_errors / p_sample_size

print(f"\nSample size : {p_sample_size} bits")
print(f"Errors      : {p_errors}")
print(f"Error rate  : {p_error_rate*100:.1f}%")
print(f"Threshold   : {ERROR_THRESHOLD*100:.0f}%")
print()

if p_error_rate > ERROR_THRESHOLD:
    print("ATTACK DETECTED — aborting key exchange!")
else:
    print("No attack detected — but Eve WAS present!")
    print(f"Eve intercepted {actually_intercepted/N_QUBITS*100:.1f}% of qubits and avoided detection.")
    print("This shows that BB84 needs a low enough threshold + enough qubits to catch subtle attackers.")
    print()
    print("In practice, this is solved by:")
    print("  1. Using more qubits (larger sample -> more statistical power)")
    print("  2. Lowering the detection threshold")
    print("  3. Privacy amplification to reduce Eve's knowledge of the final key")

=== Partial Intercept Attack (Eve intercepts 30% of qubits) ===

Alice: encoded qubits.
Eve intercepted: 26/100 qubits (26.0%)
Bob: measured qubits.
Sifted key length: 63

Sample size : 12 bits
Errors      : 0
Error rate  : 0.0%
Threshold   : 10%

No attack detected — but Eve WAS present!
Eve intercepted 26.0% of qubits and avoided detection.
This shows that BB84 needs a low enough threshold + enough qubits to catch subtle attackers.

In practice, this is solved by:
  1. Using more qubits (larger sample -> more statistical power)
  2. Lowering the detection threshold
  3. Privacy amplification to reduce Eve's knowledge of the final key
